# CE freq-only Fine-tuning

In this notebook, we fine-tune the CE-pretrained model (Cross-Entropy pretrained on awf1_freq) in a closed world scenario.

N defines the number of labeled samples that we use for fine-tuning.

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals

import warnings
warnings.filterwarnings('ignore')
import numpy as np

from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
import torch
from torch import nn
import torch.nn.functional as F
from torch import optim
from torch.autograd import Variable
import tqdm
import pickle
import argparse
from torch.cuda.amp import GradScaler, autocast

import random
import sys
import os
import collections
from sklearn.model_selection import train_test_split

## GPU Allocation

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu", 0)
kwargs = {'num_workers': 0, 'pin_memory': True} if use_cuda else {}
print (f'Device: {device}')

## Parameters

In [ ]:
batch_size = 16

## Loading the Fine-tuning Datasets

In [ ]:
DATASET = 'AWF' # 'Drift'

if DATASET == 'AWF':    
    data_path = './datasets/awf2_freq.npz' # AWF-attack
    data = np.load(data_path)
print(data.files)
# awf2_freq.npz
x_total = data['x']
y_total = data['y']

x_train_total, x_test_total, y_train_total, y_test_total = train_test_split(
    x_total, y_total, test_size=0.2, random_state=42, stratify=y_total)

x_test_sup = x_test_total
y_test_sup = y_test_total

num_classes = len(np.unique(y_train_total))
print ("Number of classes:", num_classes)

In [ ]:
print (f'Train shape: {x_train_total.shape}')
print (f'Test shape: {x_test_sup.shape}')

In [ ]:
# This function randomly samples N traces per website
def sample_traces(x, y, N):
    train_index = []
    
    for c in range(num_classes):
        idx = np.where(y == c)[0]
        idx = np.random.choice(idx, min(N, len(idx)), False)
        train_index.extend(idx)
        
    train_index = np.array(train_index)
    np.random.shuffle(train_index)
    
    x_train = x[train_index]
    y_train = y[train_index]
    
    return x_train, y_train

## Backbone Model (same as CE pretrain)

In [ ]:
class DFNet(nn.Module):
    def __init__(self, out_dim):
        super(DFNet, self).__init__()
        kernel_size = 8
        channels = [1, 32, 64, 128, 256]
        conv_stride = 1
        pool_stride = 4
        pool_size = 8
        
        self.conv1 = nn.Conv1d(1, 32, kernel_size, stride = conv_stride)
        self.conv1_1 = nn.Conv1d(32, 32, kernel_size, stride = conv_stride)
        
        self.conv2 = nn.Conv1d(32, 64, kernel_size, stride = conv_stride)
        self.conv2_2 = nn.Conv1d(64, 64, kernel_size, stride = conv_stride)
       
        self.conv3 = nn.Conv1d(64, 128, kernel_size, stride = conv_stride)
        self.conv3_3 = nn.Conv1d(128, 128, kernel_size, stride = conv_stride)
       
        self.conv4 = nn.Conv1d(128, 256, kernel_size, stride = conv_stride)
        self.conv4_4 = nn.Conv1d(256, 256, kernel_size, stride = conv_stride)
       
        
        self.batch_norm1 = nn.BatchNorm1d(32)
        self.batch_norm2 = nn.BatchNorm1d(64)
        self.batch_norm3 = nn.BatchNorm1d(128)
        self.batch_norm4 = nn.BatchNorm1d(256)
        
        self.max_pool_1 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_2 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_3 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        self.max_pool_4 = nn.MaxPool1d(kernel_size=pool_size, stride=pool_stride)
        
        self.dropout1 = nn.Dropout(p=0.1)
        self.dropout2 = nn.Dropout(p=0.1)
        self.dropout3 = nn.Dropout(p=0.1)
        self.dropout4 = nn.Dropout(p=0.1)
        
        self.fc = nn.Linear(2560, out_dim)

        
    def weight_init(self):
        for n, m in self.named_modules():
            if isinstance(m, nn.Linear) or isinstance(m, nn.Conv1d):
                print (n)
                torch.nn.init.xavier_uniform(m.weight)
                m.bias.data.zero_()
            
        
    def forward(self, inp):
        x = inp
        # ==== first block ====
        x = F.pad(x, (3,4))
        x = F.elu((self.conv1(x)))
        x = F.pad(x, (3,4))
        x = F.elu(self.batch_norm1(self.conv1_1(x)))
        x = F.pad(x, (3, 4))
        x = self.max_pool_1(x)
        x = self.dropout1(x)
        
        # ==== second block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv2(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm2(self.conv2_2(x)))
        x = F.pad(x, (3,4))
        x = self.max_pool_2(x)
        x = self.dropout2(x)
        
        # ==== third block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv3(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm3(self.conv3_3(x)))
        x = F.pad(x, (3,4))
        x = self.max_pool_3(x)
        x = self.dropout3(x)
        
        # ==== fourth block ====
        x = F.pad(x, (3,4))
        x = F.relu((self.conv4(x)))
        x = F.pad(x, (3,4))
        x = F.relu(self.batch_norm4(self.conv4_4(x)))
        x = F.pad(x, (3,4))
        x = self.max_pool_4(x)
        x = self.dropout4(x)

                
        x = x.view(x.size(0), -1)
        x = self.fc(x)
                
        return x    

## Data Loader

In [ ]:
class Data(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __getitem__(self, index):
        return self.x[index], self.y[index]
    
    def __len__(self):
        return len(self.x)

## Loading the Pre-trained Model (CE checkpoint)

**注意**: CE pretrain 的 checkpoint 是直接保存的 DFNet 权重（无 backbone. 前缀），
可以直接加载。但是 pretrain 输出的是 103 类，finetuning 也要输出 103 类，
所以 weights 命名完全一致，可以直接 strict=True 完整加载。

In [ ]:
def load_checkpoint():
    """
    Load the CE pretrained model directly.
    CE checkpoint saves DFNet(state_dict) directly (no 'backbone.' prefix),
    and both pretrain and finetune use the same num_classes (103).
    """
    model = DFNet(out_dim=num_classes).to(device)

    # ===== 加载 CE 预训练 checkpoint =====
    checkpoint = torch.load('./checkpoints/ce/WFTFC_freq_ce_epoch_100.pth.tar')

    # CE checkpoint 结构与 DFNet 完全一致，直接加载
    log = model.load_state_dict(checkpoint, strict=False)
    print(f"Missing keys (will be randomly init): {log.missing_keys}")
    print(f"Unexpected keys (ignored): {log.unexpected_keys}")
    
    return model

## Initating Test Data Loaders

In [ ]:
test_dataset_sup = Data(x_test_sup, y_test_sup)
test_loader_sup = DataLoader(test_dataset_sup, batch_size=batch_size, drop_last=True)

## Function for Train and Test

In [ ]:
def train(model, device, train_loader, optimizer):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data = data.view(data.size(0), 1, data.size(1)).float().to(device)
        target = target.to(device).long()
        
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx%100 == 0:
            print ("Loss: {:0.6f}".format(loss.item()))
    
def test(model, device, loader):
    model.eval()
    correct = 0
    temp = 0
    with torch.no_grad():
        for data, target in loader:
            data = data.view(data.size(0), 1, data.size(1)).float().to(device)
            target = target.to(device).long()
            
            output = model(data)
            output = torch.softmax(output, dim=1)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).float().sum().item()
    return correct / len(loader.dataset)

## Running for 5 Times

In [ ]:
# N defines the number of labeled samples we use to perform fine-tuning
N = 5

In [ ]:
accuracies_sup = []
for _ in range(5): # 测试5次，后面取平均值
    x_train, y_train = sample_traces(x_train_total, y_train_total, N)
    
    print ("Input size:", x_train.shape, y_train.shape)
    
    train_dataset = Data(x_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    model = load_checkpoint()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    
    
    best_acc_sup = 0
    for epoch in range(101):
        print ('Epoch: ', epoch)
        train(model, device, train_loader, optimizer)
        
        acc_sup = test(model, device, test_loader_sup)
        best_acc_sup = max(best_acc_sup, acc_sup)
        
        if epoch%10 == 0:
            print (f"Accuracy on superior dataset: {acc_sup*100:.2f}")
                
    accuracies_sup.append(best_acc_sup)
    
    
    print ('------------------------------------------------')

In [ ]:
accuracies_sup = np.array(accuracies_sup)

print (f"Test accuracy on Superior traces: avg -> {np.mean(accuracies_sup)*100:.1f}, std -> {np.std(accuracies_sup)*100:.1f}")